# Vision stages run in isolation: Gemini and Hugging Face

Runs each API stage on its own over all 260 images (OCR and the color heuristic were already run offline and are uploaded in step 3). Run the cells top to bottom. Keys are typed into a hidden prompt and are not saved in this notebook.

Expect roughly 30-40 minutes for both stages. The run is resumable: if Colab disconnects, re-run steps 1-4 and then step 6; finished images are skipped only if the CSVs are still in the session.

### 1. Clone the repo

In [ ]:
!git clone https://github.com/dhwanisinghal-sudo/Grocery-Recommender-System-Collaborative-Filtering-CV.git repo
%cd repo


### 2. Install dependencies

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install streamlit pandas numpy scikit-learn scipy pillow requests pytesseract


### 3. Upload 3 files
Select `verify_vision_stages_isolated.py`, `vision_isolated_ocr.csv` and `vision_isolated_color.csv` together. They are placed in `experiments/`.

In [ ]:
from google.colab import files
import shutil, os
up = files.upload()
os.makedirs('experiments', exist_ok=True)
for f in up:
    shutil.move(f, os.path.join('experiments', f))
    print('placed', f)
assert os.path.exists('experiments/verify_vision_stages_isolated.py'), 'script missing'


### 4. Enter API keys (hidden input)

In [ ]:
import getpass, os
g = getpass.getpass('GEMINI_API_KEY: ')
h = getpass.getpass('HF_API_TOKEN: ')
os.makedirs('.streamlit', exist_ok=True)
open('.streamlit/secrets.toml','w').write(f'GEMINI_API_KEY = "{g}"\nHF_API_TOKEN = "{h}"\n')
del g, h
print('secrets.toml written')


### 5. Smoke test (about 1 minute)
Calls each API stage once on one image and prints the raw debug messages, so a dead model name or bad key shows up now and not after 20 minutes.

In [ ]:
import sys, glob
sys.path.insert(0, '.')
import app
img = sorted(glob.glob('data/vision_test_set/Dairy/*.jpg'))[0]
data = open(img,'rb').read()
for name, fn in [('gemini', app.classify_image_gemini), ('hf', app.classify_image_hf)]:
    tags, err, dbg = fn(data)
    print(name, '->', 'OK' if tags else 'NO TAGS', dbg)


### 6. Run Gemini, then Hugging Face, each alone on all 260 images

In [ ]:
!python experiments/verify_vision_stages_isolated.py --stages gemini


In [ ]:
!python experiments/verify_vision_stages_isolated.py --stages hf


### 7. Combined summary (all four stages)

In [ ]:
!python experiments/verify_vision_stages_isolated.py --summary-only


### 8. Download results and clear keys

In [ ]:
import glob, os, zipfile
with zipfile.ZipFile('vision_isolated_results.zip','w') as z:
    for f in glob.glob('experiments/vision_isolated_*.csv'):
        z.write(f, os.path.basename(f))
from google.colab import files
files.download('vision_isolated_results.zip')
if os.path.exists('.streamlit/secrets.toml'):
    os.remove('.streamlit/secrets.toml')
print('keys removed from this session')
